Demand and Welfare
==================

**Author:** Ethan Ligon



Where household surveys earn their keep for policy: what happens to whom
when a price moves.



## Reading



Deaton is in your `reading/` folder on the hub, or [PDF](https://documents.worldbank.org/curated/en/203811547671768139/pdf/133790-PUB.pdf).

-   Deaton, chs. 4 (§4.2 on Engel curves) and 5 (all)
-   Deaton & Muellbauer (1980), "An Almost Ideal Demand System," *AER* 70:312–326
-   Lewbel (1991), "The rank of demand systems," *Econometrica* 59:711–730
-   Ligon, "Household Welfare from Disaggregate Demands" (working paper) —
    the CFE system we estimate today



## Budget shares and Engel curves



## Equivalence scales from behaviour



## Prices, and where to get them



## The rank of a demand system



## Constant Frisch Elasticity demands



## The estimating equation



## $w = -\log\lambda$ as a welfare measure



## Welfare consequences of a price change



## Budget shares



In [1]:
# Show tracebacks without the library's internal frames: the line that
# failed, and why.  Change Plain to Verbose if you ever want the rest.
%xmode Plain

import lsms_library as ll
import numpy as np, pandas as pd

ghana = ll.Country('GhanaLSS')
food = ghana.food_expenditures()
sample = ghana.sample()

wave = '2016-17'
# food_expenditures is indexed (i, t, v, j, s).  Collapse to one row per
# household-item before doing anything else: unstacking the raw index
# would ask pandas for a 13918 x 373678 array.
f = food.xs(wave, level='t').squeeze().groupby(['i', 'j']).sum()
x = f.groupby('i').sum()                     # total food expenditure
shares = f.div(x, level='i').unstack('j').fillna(0.0)
shares.shape

In [1]:
# The ten largest items by mean budget share
top = shares.mean().sort_values(ascending=False).head(10)
top.round(4)

## An Engel curve



In [1]:
import statsmodels.api as sm

# household_characteristics is a count of people by age-sex cell, indexed
# (t, v, i).  Sum across cells for household size, then drop to i alone so
# it aligns with the shares.
chars = ghana.household_characteristics().xs(wave, level='t')
hhsize = chars.sum(axis=1).groupby('i').first().rename('hhsize')

s1 = sample.xs(wave, level='t')
item = top.index[0]

d = pd.concat([shares[item].rename('w'), x.rename('x'), hhsize,
               s1.v.rename('v'), s1.weight.rename('wt')], axis=1).dropna()
d = d[(d.x > 0) & (d.hhsize > 0)]
d['logx'] = np.log(d.x.astype(float))
d['logn'] = np.log(d.hhsize.astype(float))

res = sm.WLS(d.w.astype(float),
             sm.add_constant(d[['logx', 'logn']].astype(float)),
             weights=d.wt.astype(float)
             ).fit(cov_type='cluster', cov_kwds={'groups': d.v.astype(str)})
print(res.summary().tables[1])

Note `cov_type`'cluster'`.  Session 1 established why: without it the
standard errors on =logx` are too small by roughly $\sqrt{\deff}$.



In [1]:
import matplotlib.pyplot as plt

# Nonparametric Engel curve: weighted mean share by expenditure ventile.
# A weighted mean is a ratio of two sums, so no groupby-apply is needed.
d['bin'] = pd.qcut(d.logx, 20, labels=False, duplicates='drop')
curve = (d.w * d.wt).groupby(d.bin).sum() / d.wt.groupby(d.bin).sum()
mid = d.logx.groupby(d.bin).mean()

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(mid, curve, 'o-')
ax.set_xlabel(r'$\log x$  (total food expenditure)')
ax.set_ylabel(f'budget share, {item}')
plt.show()

## Stripping quality out of unit values



In [1]:
acq = ghana.food_acquired().xs(wave, level='t')
uv = np.log((acq.Expenditure / acq.Quantity).replace([np.inf, -np.inf], np.nan)).dropna()

j0 = uv.groupby('j').size().idxmax()
u0 = uv.xs(j0, level='j').groupby('i').mean().rename('logv')

e = pd.concat([u0, d.logx, d.v], axis=1).dropna()

# Deaton's within-cluster estimator.  There are ~1000 clusters, so do not
# build the dummy matrix; by Frisch-Waugh-Lovell, demeaning within cluster
# gives the identical coefficient at a fraction of the cost.
for col in ('logv', 'logx'):
    e[col + '_d'] = e[col].astype(float) - e.groupby('v')[col].transform('mean')

within = sm.OLS(e.logv_d, e[['logx_d']]).fit(
    cov_type='cluster', cov_kwds={'groups': e.v.astype(str)})
print(f"quality elasticity psi = {within.params['logx_d']:.3f}"
      f"   (se {within.bse['logx_d']:.3f})")

A $\psi$ near zero says households of different means pay the same for this
item — it is a homogeneous good.  A large $\psi$ says "rice" in the
questionnaire is several different goods in the market.



## Estimating a CFE system



The CFE estimator needs several waves and a food classification that is
consistent across them.  Ghana's GLSS has the waves but no harmonized food
aggregation; Uganda has both, and it is the panel we use for the rest of the
workshop.  So we switch countries here, deliberately.



In [1]:
import lsms_library as ll
from cfe import Regression
import numpy as np, pandas as pd

uga = ll.Country('Uganda')
x = uga.food_expenditures().squeeze()

# Collapse the survey's fine item codes onto the harmonized aggregate
# labels: 'Matoke (bunch)', 'Matoke (heap)', ... all become 'Matoke'.
agg = (uga.categorical_mapping['harmonize_food']
          .set_index('Preferred Label')['Aggregate Label'].to_dict())
x = x.rename(index=agg, level='j')
x = x.groupby(x.index.names).sum()
print(x.index.get_level_values('j').nunique(), 'goods after aggregation')

`cfe` insists on the index being exactly $(i, t, m, j)$ — household,
period, *market*, good.  A market is a set of households facing common
prices, so the enumeration area is the natural choice; we start with a
single market, which puts all the price variation into the good-time
effects $a^j_t$.



In [1]:
y = np.log(x.replace(0, np.nan).dropna()).groupby(['i', 't', 'j']).sum()
y = pd.concat({1: y}, names=['m']).reorder_levels(['i', 't', 'm', 'j']).sort_index()

d0 = uga.household_characteristics()
d = d0.assign(
    Girls=d0[[f'F {a}' for a in ['00-03', '04-08', '09-13', '14-18']]].sum(axis=1),
    Boys =d0[[f'M {a}' for a in ['00-03', '04-08', '09-13', '14-18']]].sum(axis=1),
    Women=d0[[f'F {a}' for a in ['19-30', '31-50', '51+']]].sum(axis=1),
    Men  =d0[[f'M {a}' for a in ['19-30', '31-50', '51+']]].sum(axis=1),
)[['Girls', 'Boys', 'Women', 'Men', 'log HSize']].dropna(how='any')
d = d.groupby(['i', 't']).first()
d = pd.concat({1: d}, names=['m']).reorder_levels(['i', 't', 'm']).sort_index()

r = Regression(y=y, d=d)
beta = r.get_beta()          # a few seconds
print(len(beta), 'goods estimated')

## Frisch elasticities



In [1]:
print("least elastic:"); print(beta.sort_values().head(6).round(2).to_string())
print("\nmost elastic:"); print(beta.sort_values().tail(6).round(2).to_string())

Read those two lists before reading anything else.  The ordering is the
model's main testable implication and it is not imposed: nothing in the
estimator knows which goods are staples.  If salt and cassava do not come
out at the bottom and fruit and coffee at the top, something is wrong with
the data or with the aggregation.



In [1]:
ax = r.graph_beta(xlabel=r'Estimates of $\beta$')

We did not estimate demands for all seventy-six goods.  Items too few
households ever report cannot support a coefficient, and the estimator drops
them without being asked.  Report how many survived, because the answer
depends on the aggregation you chose.

Household composition enters through $\gamma$, one coefficient per good
per characteristic — good-specific Barten scales rather than a single
equivalence scale imposed on everything:



In [1]:
r.get_gamma().round(2).head(8)

Before trusting any of it, look at the fit:



In [1]:
import matplotlib.pyplot as plt

fit = pd.DataFrame({'actual': r.y,
                    'predicted': r.get_predicted_log_expenditures()}).dropna()
ax = fit.plot.scatter(x='predicted', y='actual', s=1, alpha=0.15, figsize=(5, 5))
lim = [fit.min().min(), fit.max().max()]
ax.plot(lim, lim, 'k--', lw=0.8)
ax.set_title(f"log expenditures, R^2 = {fit.corr().iloc[0, 1] ** 2:.2f}")
plt.show()

Now the welfare measure itself.  This one takes about half a minute.



In [1]:
w = r.get_w()                # w = -log lambda, indexed (i, t, m)
w.groupby('t').describe()[['count', 'mean', 'std']].round(3)

Estimating this took most of a minute, and sessions 4 and 5 need the same
object.  Save it rather than refitting: the pickle carries $\beta$,
$\gamma$, $w$, and the predicted expenditures together.



In [1]:
from pathlib import Path

r.predicted_expenditures()             # populate before saving
out = Path.home() / '.cache' / 'hhsurveys' / 'uganda.rgsn'
out.parent.mkdir(parents=True, exist_ok=True)
r.to_pickle(str(out))

# and to get it back:
#   import cfe
#   r = cfe.read_pickle(str(out))
print('saved', out)

## The Engel pie



An Engel curve shows one good at a time.  With a whole system estimated we
can show all of them at once: draw the predicted budget *composition* as a
pie, and let the radius be $\log x$.  Poor households at the centre, rich
at the rim.



In [1]:
import matplotlib.pyplot as plt
from matplotlib import cm

xhat = r.predicted_expenditures()
xbar = xhat.groupby(['i', 't', 'm']).sum()
p_j = ((r.y.unstack('j') > 0) + 0.).mean()      # prob. good j is purchased

lo, hi = xbar.quantile(0.05), xbar.quantile(0.95)
Y = np.geomspace(lo, hi, 60)

fig, ax = plt.subplots(figsize=(7.5, 6))
wedges = None
for k in range(len(Y) - 1, 0, -1):
    ax.set_prop_cycle('color', cm.tab20.colors)
    shares = r.expenditures(Y[k]) * p_j
    out = ax.pie(shares, radius=np.log(Y[k]) / np.log(hi), counterclock=False)
    if wedges is None:                      # keep the outermost ring
        wedges, goods = out[0], shares.index.tolist()

ax.arrow(0, 0, 1, 0, shape='full', head_width=.04, length_includes_head=True)
ax.annotate(r'$\log x$', xy=(1, 0), color='red', va='center_baseline')

# 41 goods will not fit in a legend; name the ten largest at the rim.
big = np.argsort(r.expenditures(Y[-1]).to_numpy() * p_j.to_numpy())[::-1][:10]
ax.legend([wedges[b] for b in big], [goods[b] for b in big],
          loc='center left', bbox_to_anchor=(1.02, 0.5),
          frameon=False, fontsize=8, title='largest ten, at the rim')
plt.show()

Wedges that widen as you move outward are luxuries; wedges that pinch are
necessities.  This is Engel's law, for forty-one goods simultaneously, and
it is the same information as the $\beta$ plot in a form you can show to
someone who does not know what a Frisch elasticity is.



## Exercises



1.  Estimate the Working–Leser food share equation on the *full* consumption
    aggregate rather than food alone, and test whether $\beta$ differs
    between urban and rural households.
2.  Implement the Engel equivalence scale: find the expenditure ratio that
    equates the predicted food share of a household with two adults and two
    children to that of a household with two adults.  Compare with the
    $\theta$-scale you used in session 2.
3.  Compute the first-order welfare cost of a 20% rise in the price of the
    largest food item, by consumption decile.  Who loses most, in cedis and
    as a share of expenditure?  Are those the same households?
4.  Re-estimate the CFE system using the enumeration area as the market
    $m$ rather than a single national market.  How much do the
    $\beta_j$ move?  What have you assumed away by using one market?
5.  Regress $w$ on $\log$ total food expenditure.  The two are
    measuring the same thing in principle; how closely do they agree, and
    which households do they disagree about?

